# Grain Futures Strategy Research Report

This notebook is the cleaned research report for the grain-futures project. It shows what was tested, why it was tested, the score tables, regime/year-period behavior, and which strategy is selected for each use case.

Core convention:
- Traded baseline is outright front-month futures in CORN, SOYABEAN, WHEAT_SRW, and WHEAT_HRW.
- Calendar/inter-commodity spreads are tested separately as tradable experimental sleeves.
- Train/test split is 2018-01-01.
- Validation window is 2016-01-01 to 2017-12-31.
- 2022 Ukraine war is not in the supplied train_set, which ends 2020-12-31; it is flagged as a required future holdout regime.

In [2]:
from __future__ import print_function

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from grain_futures_strategy import (
    COMMODITIES,
    OUTRIGHT_CORE_FEATURES,
    OUTRIGHT_PHYSICAL_FEATURES,
    run_research_pipeline,
    run_spread_experiment,
    run_holding_period_experiment,
    run_filter_sleeve_experiment,
    run_cost_margin_experiment,
    run_final_strategy_selection,
    run_observable_regime_weight_experiment,
    skip_rebalance_positions,
    multi_condition_filter_positions,
    backtest_positions,
    split_performance,
    period_performance,
)

%matplotlib inline
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 80)

In [ ]:
import importlib
import grain_futures_strategy as gfs
gfs = importlib.reload(gfs)

COMMODITIES = gfs.COMMODITIES
OUTRIGHT_CORE_FEATURES = gfs.OUTRIGHT_CORE_FEATURES
OUTRIGHT_PHYSICAL_FEATURES = gfs.OUTRIGHT_PHYSICAL_FEATURES
run_research_pipeline = gfs.run_research_pipeline
run_spread_experiment = gfs.run_spread_experiment
run_holding_period_experiment = gfs.run_holding_period_experiment
run_filter_sleeve_experiment = gfs.run_filter_sleeve_experiment
run_cost_margin_experiment = gfs.run_cost_margin_experiment
run_final_strategy_selection = gfs.run_final_strategy_selection
run_observable_regime_weight_experiment = gfs.run_observable_regime_weight_experiment
skip_rebalance_positions = gfs.skip_rebalance_positions
multi_condition_filter_positions = gfs.multi_condition_filter_positions
backtest_positions = gfs.backtest_positions
split_performance = gfs.split_performance
period_performance = gfs.period_performance


## Configuration And Data Windows

The validation window is used to reduce OOS threshold tuning. The test/OOS period starts on 2018-01-01. The named regime periods are based on grain-market events available in the supplied 2010-2020 sample.

In [3]:
DATA_DIR = 'train_set'
SPLIT_DATE = '2018-01-01'
VALIDATION_START = '2016-01-01'
VALIDATION_END = '2017-12-31'
COST_PER_LOT = 0.0

assert os.path.isdir(DATA_DIR), 'Expected train_set directory next to this notebook'

## Run Reproducible Experiments

This cell runs the cleaned module-level experiments:
- Main outright strategy pipeline.
- Tradable spread experiment.
- Holding-period sweep.
- Filter/smoothing/carry-storage sleeve tests.

In [4]:
results = run_research_pipeline(DATA_DIR, split_date=SPLIT_DATE, cost_per_lot=COST_PER_LOT)
spread_results = run_spread_experiment(DATA_DIR, split_date=SPLIT_DATE, cost_per_lot=COST_PER_LOT)
holding_results = run_holding_period_experiment(DATA_DIR, split_date=SPLIT_DATE, cost_per_lot=COST_PER_LOT)
filter_results = run_filter_sleeve_experiment(DATA_DIR, split_date=SPLIT_DATE)
cost_margin_results = run_cost_margin_experiment(DATA_DIR, split_date=SPLIT_DATE)
final_strategy_results = run_final_strategy_selection(DATA_DIR, split_date=SPLIT_DATE)
regime_weight_results = run_observable_regime_weight_experiment(DATA_DIR, split_date=SPLIT_DATE)

summary = results['summary']
summary

,dataset,rows,columns,start,end,missing_cells,columns_list
0,adj1,2866,4,2010-01-04,2020-12-31,0,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"
1,adj2,2866,4,2010-01-04,2020-12-31,0,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"
2,cgl_crush,1858,2,2015-12-01,2020-12-31,0,"processed, planned"
3,cgl_inv,2771,4,2010-01-04,2020-12-31,4,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"
4,cot_mm,581,4,2010-01-05,2020-12-29,28,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"
5,cot_pm_oi,576,4,2010-01-05,2020-12-29,8,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"
6,inventories,835,4,2005-01-04,2020-12-29,2,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"
7,receipts,1127,4,2016-07-12,2020-12-31,0,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"
8,unadj1,2866,4,2010-01-04,2020-12-31,0,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"
9,unadj2,2866,4,2010-01-04,2020-12-31,0,"CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW"


## Cargill Data Usage

The main outright Ridge strategy uses both Cargill datasets:
- `cgl_inv` -> Cargill inventory level/change.
- `cgl_crush` -> processed, planned, surprise, utilization.

Some later diagnostic factor/spread experiments use `cgl_inv` but not `cgl_crush`, because the crush file is aggregate processed/planned data rather than a clean spread-leg input. A tradable soybean crush spread was not tested because soybean meal/oil futures are not in this dataset.

In [5]:
feature_usage = pd.DataFrame({
    'feature_block': ['core', 'physical'],
    'features': [', '.join(OUTRIGHT_CORE_FEATURES), ', '.join(OUTRIGHT_PHYSICAL_FEATURES)],
    'uses_cgl_inv': [False, True],
    'uses_cgl_crush': [False, True],
})
feature_usage

,feature_block,features,uses_cgl_inv,uses_cgl_crush
0,core,"mom_60, rev_5, curve_spread, curve_ratio, cot_...",False,False
1,physical,"public_inventory_change, receipts_change, cgl_...",True,True


## Experiment Scoreboard

This table collects the major experiments and their train/validation/test scores. The key columns requested are included:
`validation_sharpe`, `is_sharpe`, `hold_days`, `oos_sharpe`, `oos_pnl`, `full_sharpe`, `max_dd`, and `turnover`.

Interpretation note: some older experiments were research diagnostics, so their validation score is blank when not run under the fixed 2016-2017 validation protocol.

In [6]:
def score_from_split(name, metrics, rationale, selected_for, hold_days=1, validation_sharpe=np.nan):
    return {
        'experiment': name,
        'rationale': rationale,
        'selected_for': selected_for,
        'hold_days': hold_days,
        'validation_sharpe': validation_sharpe,
        'is_sharpe': metrics.loc['sharpe', 'in_sample'],
        'oos_sharpe': metrics.loc['sharpe', 'out_of_sample'],
        'oos_pnl': metrics.loc['total_pnl', 'out_of_sample'],
        'full_sharpe': metrics.loc['sharpe', 'full_period'],
        'max_dd': metrics.loc['max_drawdown', 'full_period'],
        'turnover': metrics.loc['avg_daily_turnover', 'full_period'],
    }

rows = []
rows.append(score_from_split('Old broad one-day Ridge', results['broad_metrics'], 'All features in one one-day Ridge model; first broad baseline.', 'Rejected: severe overfit / negative OOS'))
rows.append(score_from_split('Static unfiltered two-block Ridge', results['unfiltered_metrics'], 'Separate core and physical Ridge blocks without edge filter.', 'Rejected: useful but weaker control than edge-filtered'))
rows.append(score_from_split('Static edge-filtered two-block Ridge', results['model_metrics'], 'Core + physical Ridge with prediction-dispersion edge filter.', 'Main tradable candidate', 1, filter_results['filter_sleeve_table'].set_index('experiment').loc['Baseline static edge-filtered', 'validation_sharpe']))
rows.append(score_from_split('Annual walk-forward Ridge', results['walk_forward_metrics'], 'Annual expanding retrain; 20-day target; cleaner anti-overfit check.', 'Primary anti-overfit validation'))
rows.append(score_from_split('60-day momentum baseline', results['baseline_metrics'], 'Simple cross-sectional momentum benchmark.', 'Benchmark only'))
rows.extend(filter_results['filter_sleeve_table'].to_dict('records'))
rows.append(score_from_split('Calendar spreads only', spread_results['calendar_metrics'], 'Tradable front-vs-second calendar spread sleeve.', 'Rejected: too weak'))
rows.append(score_from_split('Inter-commodity spreads only', spread_results['intercommodity_metrics'], 'Vol-hedged relative-value grain pairs.', 'Useful as overlay, not standalone replacement'))
rows.append(score_from_split('80% outright + 20% inter-commodity overlay', spread_results['inter_overlay_20_metrics'], 'Diversification overlay using inter-commodity spread sleeve.', 'Extension candidate: better Sharpe/drawdown, lower OOS PnL'))
rows.append(score_from_split('Naive all-instrument book', spread_results['combined_metrics'], 'Outrights, calendar spreads, and inter-commodity spreads ranked together.', 'Rejected: dilutes stronger outright signal'))
rows.extend([
    {'experiment':'Best sklearn ML: shallow Gradient Boosting','rationale':'Nonlinear ML with annual walk-forward, median imputation, missing indicators.','selected_for':'Rejected: close but worse than Ridge WF and higher overfit risk','hold_days':1,'validation_sharpe':np.nan,'is_sharpe':0.047,'oos_sharpe':1.274,'oos_pnl':10450.0,'full_sharpe':0.633,'max_dd':-3164.0,'turnover':0.392},
    {'experiment':'Online literature factor: price + term + HP + storage','rationale':'Momentum, term structure, hedging pressure, and storage factor basket.','selected_for':'Rejected: high OOS but regime-fragile full-period drawdown','hold_days':1,'validation_sharpe':np.nan,'is_sharpe':-0.089,'oos_sharpe':1.468,'oos_pnl':11834.0,'full_sharpe':0.265,'max_dd':-11814.0,'turnover':0.362},
])
experiment_scores = pd.DataFrame(rows)
cols = ['experiment', 'selected_for', 'hold_days', 'validation_sharpe', 'is_sharpe', 'oos_sharpe', 'oos_pnl', 'full_sharpe', 'max_dd', 'turnover', 'rationale']
experiment_scores = experiment_scores[cols].drop_duplicates(subset=['experiment'], keep='last')
experiment_scores.sort_values(['oos_sharpe', 'full_sharpe'], ascending=False).round(3)

,experiment,selected_for,hold_days,validation_sharpe,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,max_dd,turnover,rationale
7,Multi-condition opportunity filter,Best crisis-regime and Sharpe candidate,1,2.480,1.573,2.098,7555.641,1.736,-3405.688,0.338,"Trade only when prediction, curve, and momentu..."
6,2-day skip-rebalance,Best OOS PnL / lower-turnover execution variant,2,0.610,0.862,1.730,11637.135,1.087,-4111.012,0.161,Hold positions for two trading days to reduce ...
2,Static edge-filtered two-block Ridge,Main tradable candidate,1,0.617,0.930,1.498,10303.820,1.077,-4648.140,0.234,Core + physical Ridge with prediction-dispersi...
5,Baseline static edge-filtered,Baseline tradable candidate,1,0.617,0.930,1.498,10303.820,1.077,-4648.140,0.234,Two-block Ridge plus prediction-dispersion edg...
15,Online literature factor: price + term + HP + ...,Rejected: high OOS but regime-fragile full-per...,1,NaN,-0.089,1.468,11834.000,0.265,-11814.000,0.362,"Momentum, term structure, hedging pressure, an..."
9,20% carry/storage overlay,Rejected: weak full-period robustness,1,0.556,0.574,1.458,9091.960,0.778,-4170.089,0.177,Add simple carry/backwardation and inventory-d...
12,80% outright + 20% inter-commodity overlay,"Extension candidate: better Sharpe/drawdown, l...",1,NaN,0.273,1.416,10762.702,0.836,-3615.290,0.244,Diversification overlay using inter-commodity ...
3,Annual walk-forward Ridge,Primary anti-overfit validation,1,NaN,0.143,1.347,11579.455,0.738,-4000.501,0.234,Annual expanding retrain; 20-day target; clean...
14,Best sklearn ML: shallow Gradient Boosting,Rejected: close but worse than Ridge WF and hi...,1,NaN,0.047,1.274,10450.000,0.633,-3164.000,0.392,"Nonlinear ML with annual walk-forward, median ..."
8,Position smoothing HL2,Rejected: turnover reduction only,1,0.613,0.829,1.097,7860.488,0.895,-3666.552,0.090,EWMA smooth positions to reduce turnover.


## Signal Scores And Feature Attribution

The Ridge coefficients are shown as signal scores. Because the input features are z-scored, larger absolute coefficients mean the model relied more heavily on that signal in-sample. The physical block includes both public data and Cargill-specific features.

In [7]:
core_scores = results['coefficients']['core'].abs().mean(axis=1).sort_values(ascending=False).to_frame('mean_abs_coef')
phys_scores = results['coefficients']['physical'].abs().mean(axis=1).sort_values(ascending=False).to_frame('mean_abs_coef')

print('Core signal scores')
display(core_scores.round(3))

print('Physical/Cargill signal scores')
display(phys_scores.round(3))

Core signal scores


,mean_abs_coef
curve_spread,171.577
cot_mm_level,171.363
curve_ratio,147.173
cot_pm_oi_level,125.930
mom_60,76.982
rev_5,37.978


Physical/Cargill signal scores


,mean_abs_coef
cgl_inventory_change,19.853
public_inventory_change,18.508
receipts_change,14.393
crush_utilization,12.614
crush_surprise,11.446


## Regime / Year-Period Scoreboard

The table below scores selected strategies by named market regime. The selected strategy per regime is picked by highest Sharpe inside that period. The 2022 Ukraine war is outside the current dataset and should be tested when the data extension is available.

How to interpret regime switch here:
- The named periods are historical event labels used for diagnostics after the fact. We know them from dated market events such as droughts, trade-war tariff dates, planting floods, and COVID.
- The `picked_strategy` column is therefore not a live switching rule by itself; it says which tested strategy would have worked best inside each known historical regime.
- A live regime switch must come from observable rules available at the time. In this notebook, the live-style switch is the multi-condition opportunity filter: prediction dispersion, curve dispersion, and momentum dispersion are all computed with expanding, lagged thresholds.
- The Ukraine war is mentioned as future work because it starts in 2022, outside the current train_set ending 2020-12-31.

Practical switching rule used in this notebook:
- Do not switch strategies because a row says `picked_strategy` for a historical period. Those rows are evaluation labels, not signals.
- If the goal is one deployable strategy, use one pre-selected variant: static edge-filtered for main PnL, 2-day skip-rebalance for lower-turnover/high-PnL execution, or multi-condition filter for crisis-aware/high-Sharpe exposure.
- If a live switch is desired, it must be pre-registered before the holdout. The current live-testable switch is: trade the multi-condition filter when prediction, curve, and momentum dispersion clear their expanding thresholds; otherwise stay flat or use the baseline static book, depending on the chosen mandate.
- The table you pasted can be used to understand which historical regimes favored which strategy, but it should not be used to select strategies after seeing the regime label.


In [8]:
selected_positions = {
    'Static edge-filtered': results['model_positions'],
    '2-day skip-rebalance': skip_rebalance_positions(results['model_positions'], 2),
    'Multi-condition filter': multi_condition_filter_positions(results['predictions'], results['futures_pnl'], results['feature_panels'], 0.40, 0.40, 0.40)[0],
    'Annual walk-forward Ridge': results['walk_forward_positions'],
}
regime_rows = []
for strategy_name, positions in selected_positions.items():
    bt, _ = backtest_positions(positions, results['futures_pnl'], COST_PER_LOT)
    tbl = period_performance(bt)
    tbl['strategy'] = strategy_name
    regime_rows.append(tbl)
regime_scores = pd.concat(regime_rows, ignore_index=True)
regime_cols = ['period', 'start', 'end', 'reason', 'strategy', 'days', 'total_pnl', 'sharpe', 'max_drawdown', 'hit_rate']
regime_scores = regime_scores[regime_cols]
picked_by_regime = (regime_scores.dropna(subset=['sharpe']).sort_values(['period', 'sharpe'], ascending=[True, False]).groupby('period').head(1)[['period', 'start', 'end', 'reason', 'strategy', 'sharpe', 'total_pnl', 'max_drawdown']].rename(columns={'strategy': 'picked_strategy'}))
print('Picked strategy by regime')
display(picked_by_regime.round(3))
print('All selected-strategy regime scores')
display(regime_scores.round(3))

Picked strategy by regime


,period,start,end,reason,picked_strategy,sharpe,total_pnl,max_drawdown
21,2019 prevented planting floods,2019-05-01,2019-07-31,Wet spring and Midwest flooding delayed corn a...,Multi-condition filter,3.449,966.011,-620.708
22,COVID demand shock,2020-02-24,2020-06-30,COVID restrictions reduced gasoline/ethanol de...,Multi-condition filter,3.556,1125.889,-412.476
15,COVID recovery/China buying,2020-07-01,2020-12-31,Recovery phase with stronger Chinese buying an...,2-day skip-rebalance,4.084,7049.419,-1033.911
18,Crimea/Black Sea shock,2014-02-15,2014-05-31,Ukraine/Crimea crisis raised Black Sea wheat a...,Multi-condition filter,5.689,3238.039,-1086.140
19,Low-price abundant supply,2014-06-01,2017-12-31,Post-drought supply rebuild and generally lowe...,Multi-condition filter,1.377,5428.714,-2882.618
8,Russian drought/export ban shock,2010-07-01,2011-06-30,"Russian heat wave, drought, and grain export b...",2-day skip-rebalance,1.560,2878.131,-2636.752
1,US drought rally/retrace,2012-06-01,2013-05-31,Historic US drought drove corn/soybean/wheat p...,Static edge-filtered,1.851,2956.616,-1434.793
20,US-China trade war,2018-07-06,2020-01-15,Tariff escalation hit US soybean demand until ...,Multi-condition filter,1.496,2692.882,-1296.823


All selected-strategy regime scores


,period,start,end,reason,strategy,days,total_pnl,sharpe,max_drawdown,hit_rate
0,Russian drought/export ban shock,2010-07-01,2011-06-30,"Russian heat wave, drought, and grain export b...",Static edge-filtered,89.0,1992.697,1.038,-2552.852,0.494
1,US drought rally/retrace,2012-06-01,2013-05-31,Historic US drought drove corn/soybean/wheat p...,Static edge-filtered,94.0,2956.616,1.851,-1434.793,0.564
2,Crimea/Black Sea shock,2014-02-15,2014-05-31,Ukraine/Crimea crisis raised Black Sea wheat a...,Static edge-filtered,75.0,3755.033,2.335,-2305.309,0.520
3,Low-price abundant supply,2014-06-01,2017-12-31,Post-drought supply rebuild and generally lowe...,Static edge-filtered,743.0,1366.315,0.145,-4443.154,0.482
4,US-China trade war,2018-07-06,2020-01-15,Tariff escalation hit US soybean demand until ...,Static edge-filtered,264.0,-117.811,-0.045,-2602.034,0.519
5,2019 prevented planting floods,2019-05-01,2019-07-31,Wet spring and Midwest flooding delayed corn a...,Static edge-filtered,43.0,818.152,2.140,-620.708,0.628
6,COVID demand shock,2020-02-24,2020-06-30,COVID restrictions reduced gasoline/ethanol de...,Static edge-filtered,83.0,1343.525,1.543,-642.420,0.530
7,COVID recovery/China buying,2020-07-01,2020-12-31,Recovery phase with stronger Chinese buying an...,Static edge-filtered,117.0,6714.594,3.844,-1111.437,0.581
8,Russian drought/export ban shock,2010-07-01,2011-06-30,"Russian heat wave, drought, and grain export b...",2-day skip-rebalance,88.0,2878.131,1.560,-2636.752,0.523
9,US drought rally/retrace,2012-06-01,2013-05-31,Historic US drought drove corn/soybean/wheat p...,2-day skip-rebalance,100.0,1567.940,0.919,-2411.550,0.550


## Holding-Period Experiment

Rationale: the model predicts multi-day returns, so we tested whether holding positions for 2 or 5 days improves results and reduces turnover. The 2-day skip-rebalance version improves the static strategy, but the walk-forward anti-overfit model does not confirm the same benefit.

In [9]:
hp = holding_results['holding_period_table']
show_cols = ['strategy', 'method', 'hold_days', 'is_sharpe', 'oos_sharpe', 'oos_pnl', 'full_sharpe', 'max_drawdown', 'turnover']
print('Top holding-period variants')
display(holding_results['best_by_oos_sharpe'][show_cols].round(3))
print('Full holding-period sweep')
display(hp[show_cols].round(3))

Top holding-period variants


,strategy,method,hold_days,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,max_drawdown,turnover
1,Static edge-filtered,skip-rebalance,2,0.862,1.730,11637.135,1.087,-4111.012,0.161
0,Static edge-filtered,daily,1,0.930,1.498,10303.820,1.077,-4648.140,0.234
2,Static edge-filtered,skip-rebalance,3,0.792,1.441,9693.906,0.956,-4408.601,0.130
5,Static edge-filtered,staggered,2,0.903,1.413,9692.062,1.034,-4108.369,0.175
11,Static unfiltered,skip-rebalance,3,0.789,1.385,12003.318,0.918,-7896.034,0.136
27,Walk-forward,daily,1,0.143,1.347,11579.455,0.738,-4000.501,0.234
18,WF edge-filtered,daily,1,0.143,1.331,11347.417,0.728,-4000.501,0.235
6,Static edge-filtered,staggered,3,0.857,1.320,9075.460,0.975,-4006.316,0.143
23,WF edge-filtered,staggered,2,0.270,1.281,10666.425,0.764,-3917.851,0.159
28,Walk-forward,skip-rebalance,2,0.254,1.276,10836.759,0.755,-4222.771,0.160


Full holding-period sweep


,strategy,method,hold_days,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,max_drawdown,turnover
0,Static edge-filtered,daily,1,0.930,1.498,10303.820,1.077,-4648.140,0.234
1,Static edge-filtered,skip-rebalance,2,0.862,1.730,11637.135,1.087,-4111.012,0.161
2,Static edge-filtered,skip-rebalance,3,0.792,1.441,9693.906,0.956,-4408.601,0.130
3,Static edge-filtered,skip-rebalance,5,0.886,1.213,7994.266,0.965,-4097.036,0.101
4,Static edge-filtered,skip-rebalance,10,0.634,0.633,4089.420,0.629,-4725.472,0.069
5,Static edge-filtered,staggered,2,0.903,1.413,9692.062,1.034,-4108.369,0.175
6,Static edge-filtered,staggered,3,0.857,1.320,9075.460,0.975,-4006.316,0.143
7,Static edge-filtered,staggered,5,0.937,1.110,7632.714,0.978,-3757.417,0.105
8,Static edge-filtered,staggered,10,0.874,0.937,6423.756,0.886,-4186.100,0.065
9,Static unfiltered,daily,1,0.558,1.087,9461.345,0.673,-7449.207,0.240


## Spread Instrument Experiment

Rationale: curve variables were initially predictors only. This test treats calendar spreads and inter-commodity spreads as tradable instruments with their own PnL targets.

Selection:
- Calendar spreads: rejected, too weak.
- Inter-commodity spreads: useful as modest overlay.
- Naive all-instrument ranking: rejected, dilutes the outright edge.

In [10]:
spread_rows = []
for name, metrics in [
    ('outright_wf', spread_results['outright_metrics']),
    ('calendar_only', spread_results['calendar_metrics']),
    ('intercommodity_only', spread_results['intercommodity_metrics']),
    ('all_spreads', spread_results['spread_metrics']),
    ('naive_all_instruments', spread_results['combined_metrics']),
    ('80pct_outright_20pct_inter', spread_results['inter_overlay_20_metrics']),
    ('70pct_outright_30pct_inter', spread_results['inter_overlay_30_metrics']),
    ('50pct_outright_50pct_all_spread', spread_results['spread_overlay_50_metrics']),
]:
    spread_rows.append({
        'experiment': name,
        'is_sharpe': metrics.loc['sharpe', 'in_sample'],
        'oos_sharpe': metrics.loc['sharpe', 'out_of_sample'],
        'oos_pnl': metrics.loc['total_pnl', 'out_of_sample'],
        'full_sharpe': metrics.loc['sharpe', 'full_period'],
        'max_dd': metrics.loc['max_drawdown', 'full_period'],
        'turnover': metrics.loc['avg_daily_turnover', 'full_period'],
    })
spread_score_table = pd.DataFrame(spread_rows)
spread_score_table.sort_values('oos_sharpe', ascending=False).round(3)

,experiment,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,max_dd,turnover
5,80pct_outright_20pct_inter,0.273,1.416,10762.702,0.836,-3615.290,0.244
6,70pct_outright_30pct_inter,0.338,1.415,10354.325,0.865,-3424.721,0.248
0,outright_wf,0.143,1.347,11579.455,0.738,-4000.501,0.234
7,50pct_outright_50pct_all_spread,0.510,1.263,7374.610,0.890,-2521.605,0.246
2,intercommodity_only,0.504,0.768,7495.690,0.625,-4295.690,0.281
4,naive_all_instruments,0.579,0.651,4115.262,0.616,-3017.303,0.254
3,all_spreads,0.817,0.550,3169.766,0.682,-3093.861,0.258
1,calendar_only,-0.012,0.176,188.031,0.068,-1543.958,0.255


## Filter, Smoothing, And Carry/Storage Sleeve Tests

Rationale: the baseline was weak during the US-China trade-war period. These tests ask whether we can avoid bad regimes or add a defensive carry/storage sleeve.

Selection:
- Multi-condition opportunity filter is the best Sharpe/crisis candidate.
- 2-day skip-rebalance is best for OOS PnL plus lower turnover.
- Position smoothing and carry/storage sleeve are rejected as primary improvements.

In [11]:
filter_cols = ['experiment', 'selected_for', 'hold_days', 'validation_sharpe', 'is_sharpe', 'oos_sharpe', 'oos_pnl', 'full_sharpe', 'max_dd', 'turnover', 'us_china_trade_war_sharpe', 'us_china_trade_war_pnl', 'rationale']
filter_results['filter_sleeve_table'][filter_cols].round(3)

,experiment,selected_for,hold_days,validation_sharpe,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,max_dd,turnover,us_china_trade_war_sharpe,us_china_trade_war_pnl,rationale
0,Baseline static edge-filtered,Baseline tradable candidate,1,0.617,0.930,1.498,10303.820,1.077,-4648.140,0.234,-0.045,-117.811,Two-block Ridge plus prediction-dispersion edg...
1,2-day skip-rebalance,Best OOS PnL / lower-turnover execution variant,2,0.610,0.862,1.730,11637.135,1.087,-4111.012,0.161,0.517,1282.322,Hold positions for two trading days to reduce ...
2,Multi-condition opportunity filter,Best crisis-regime and Sharpe candidate,1,2.480,1.573,2.098,7555.641,1.736,-3405.688,0.338,1.496,2692.882,"Trade only when prediction, curve, and momentu..."
3,Position smoothing HL2,Rejected: turnover reduction only,1,0.613,0.829,1.097,7860.488,0.895,-3666.552,0.090,0.047,135.012,EWMA smooth positions to reduce turnover.
4,20% carry/storage overlay,Rejected: weak full-period robustness,1,0.556,0.574,1.458,9091.960,0.778,-4170.089,0.177,0.094,223.417,Add simple carry/backwardation and inventory-d...


## Cost, Holding Cost, And Margin Stress Test

Rationale: the earlier results were shown with zero transaction costs. This section tests whether the selected strategies survive more realistic implementation assumptions.

Assumption source and interpretation:
- CME grain contracts are full-size 5,000 bushel futures, and a 1/4 cent tick is $12.50 per contract for corn/soy/wheat-style contracts.
- CME margins are performance bonds and vary through time by product and volatility; this notebook uses transparent approximations rather than claiming exact broker margin.
- Market-assumption trade cost is $8.75 per one-way lot traded: roughly half a tick plus estimated commissions/fees.
- Holding cost is margin funding cost: estimated margin used times annual funding rate divided by 252.
- CDF means cost drag fraction: total cost divided by absolute gross PnL over active days.

Cases:
- zero_cost_no_margin_cap: research baseline.
- market_assumption: trade cost plus 5% annual holding/funding cost.
- market_assumption_margin_cap: same costs plus a 2,500 USD aggregate margin budget.
- stress_cost_margin_cap: wider execution cost, 8% funding, same margin budget.

In [12]:
cost_cols = [
    'strategy', 'case', 'CDF', 'trade_cost', 'holding_cost', 'total_cost',
    'avg_margin_used', 'avg_margin_scale', 'is_sharpe', 'oos_sharpe',
    'oos_pnl', 'full_sharpe', 'max_dd', 'turnover'
]
cost_margin_table = cost_margin_results['cost_margin_table']
cost_margin_table[cost_cols].round(3)

,strategy,case,CDF,trade_cost,holding_cost,total_cost,avg_margin_used,avg_margin_scale,is_sharpe,oos_sharpe,oos_pnl,full_sharpe,max_dd,turnover
0,Static edge-filtered,zero_cost_no_margin_cap,0.000,0.000,0.000,0.000,2458.359,1.000,0.930,1.498,10303.820,1.077,-4648.140,0.234
1,Static edge-filtered,market_assumption,0.173,3840.082,913.105,4753.186,2458.359,1.000,0.762,1.255,8637.787,0.890,-5027.694,0.234
2,Static edge-filtered,market_assumption_margin_cap,0.180,3680.552,876.815,4557.367,2360.655,0.967,0.728,1.225,7900.533,0.855,-4993.376,0.225
3,Static edge-filtered,stress_cost_margin_cap,0.305,6309.518,1402.904,7712.422,2360.655,0.967,0.611,1.055,6806.739,0.725,-5251.981,0.225
4,2-day skip-rebalance,zero_cost_no_margin_cap,0.000,0.000,0.000,0.000,2456.829,1.000,0.862,1.730,11637.135,1.087,-4111.012,0.161
5,2-day skip-rebalance,market_assumption,0.130,2648.737,913.999,3562.736,2456.829,1.000,0.733,1.550,10438.853,0.945,-4424.913,0.161
6,2-day skip-rebalance,market_assumption_margin_cap,0.135,2546.515,877.853,3424.367,2359.668,0.966,0.695,1.539,9695.175,0.910,-4384.719,0.155
7,2-day skip-rebalance,stress_cost_margin_cap,0.228,4365.454,1404.565,5770.018,2359.668,0.966,0.605,1.414,8914.686,0.812,-4597.211,0.155
8,Multi-condition filter,zero_cost_no_margin_cap,0.000,0.000,0.000,0.000,2491.094,1.000,1.573,2.098,7555.641,1.736,-3405.688,0.338
9,Multi-condition filter,market_assumption,0.143,2219.704,371.193,2590.896,2491.094,1.000,1.346,1.801,6502.007,1.488,-3728.537,0.338


Cost/margin read-through:
- 2-day skip-rebalance remains the best OOS PnL candidate after market costs.
- Multi-condition filter remains the best Sharpe and crisis-avoidance candidate after market costs.
- Annual walk-forward Ridge is more cost-sensitive because its edge is thinner and turnover remains meaningful.
- Margin caps reduce exposure slightly in this dataset because average margin use is already close to the 2,500 USD budget.

## Observable Three-Sleeve Regime Weighting Test

The named drought, flood, COVID, and trade-war regimes are useful diagnostics, but they are not live trading signals. This test uses three sleeves and computes weights only from lagged, observable state variables:
- static edge-filtered sleeve,
- 2-day skip-rebalance sleeve,
- multi-condition opportunity-filter sleeve.

The regime state uses lagged realized grain volatility and lagged expanding opportunity percentiles. It does not use the historical period label. This makes it a live-testable regime-weighting candidate rather than an ex-post period switch.


In [14]:
regime_weight_results = run_observable_regime_weight_experiment(DATA_DIR, split_date=SPLIT_DATE)
regime_weight_results['assumptions']


         assumption                                                                                                         value
            sleeves                                            static edge-filtered, 2-day skip-rebalance, multi-condition filter
      regime_inputs                                                 lagged realized grain volatility and lagged opportunity score
   no_named_regimes                                                           drought/COVID/trade-war labels are diagnostics only
       vol_low_rule                                                                           ewm_vol / expanding_long_vol < 0.70
      vol_high_rule                                                                           ewm_vol / expanding_long_vol > 1.30
    crisis_cap_rule                                                      if vol_ratio > 2.00, vol-scaled variant applies 0.50 cap
          cost_case                                                                       

In [15]:
regime_weight_results['weight_summary'].round(3)


                                 strategy  avg_static_weight  avg_skip_weight  avg_multi_condition_weight  avg_vol_scale
            Observable three-sleeve blend              0.391            0.299                        0.31          1.000
Observable three-sleeve blend + vol scale              0.391            0.299                        0.31          1.144


In [ ]:
# Reload the project module so notebook edits pick up the latest helper columns.
import importlib
import grain_futures_strategy as gfs
gfs = importlib.reload(gfs)
regime_weight_results = gfs.run_observable_regime_weight_experiment(DATA_DIR, split_date=SPLIT_DATE)

regime_weight_cols = [
    'strategy', 'case', 'trade_cost_per_lot', 'holding_cost_rate',
    'is_sharpe', 'oos_sharpe', 'oos_pnl', 'full_sharpe',
    'full_pnl', 'full_hit_rate', 'max_dd', 'turnover',
    'total_turnover_lots', 'margin_dollar_days',
    'trade_cost', 'expected_trade_cost',
    'holding_cost', 'expected_holding_cost', 'total_cost', 'CDF'
]
regime_weight_table = regime_weight_results['regime_weight_table']
available_regime_weight_cols = [c for c in regime_weight_cols if c in regime_weight_table.columns]
missing_regime_weight_cols = [c for c in regime_weight_cols if c not in regime_weight_table.columns]
if missing_regime_weight_cols:
    print('Missing columns after reload:', missing_regime_weight_cols)
    print('Available columns:', list(regime_weight_table.columns))
regime_weight_table[available_regime_weight_cols].round(3)


Read-through:
- The same unit cost assumptions are used for every strategy: $8.75 per one-way lot and 5% annual margin funding.
- Total dollar costs differ because each sleeve has different turnover and margin usage. The `total_turnover_lots` and `margin_dollar_days` columns reconcile the cost math.
- The observable three-sleeve blend uses all three strategies and assigns average weights of roughly 39% static, 30% 2-day skip, and 31% multi-condition.
- It is not selected as the final strategy because, after market costs, its OOS Sharpe is below the fixed 50/50 skip + multi blend.
- The volatility-scaled variant increases OOS PnL, but it also raises turnover and drawdown and does not improve Sharpe.
- Because the regime weights are based only on lagged volatility/opportunity data, this test is lower-overfit than switching by named drought/COVID/trade-war labels. It still needs the untouched 2021-February 2026 holdout before promotion.


## No-Fit Cargill-Aware Reversion Rule

This is the most conservative response to the concern that Ridge coefficients can overfit. It removes fitted regression coefficients entirely. The signal is fixed: short-term mean reversion plus a small fixed physical-pressure tilt using public inventory/receipts, Cargill inventory, and soybean-specific Cargill crush pressure. Positions are volatility-targeted using lagged realized strategy PnL. This still can be overfit through research selection, so it must be frozen before the 2021-February 2026 holdout.


In [ ]:
from grain_futures_strategy import run_no_fit_reversion_strategy

no_fit_results = run_no_fit_reversion_strategy(DATA_DIR, split_date=SPLIT_DATE)
print("No-fit strategy assumptions")
display(no_fit_results["assumptions"])
print("No-fit strategy zero-cost metrics")
display(no_fit_results["zero_cost_metrics"].round(3))
print("No-fit strategy cost-adjusted metrics")
display(no_fit_results["cost_adjusted_metrics"].round(3))


No-fit rule read-through:
- Strength: no learned Ridge coefficients, uses both Cargill datasets, high zero-cost OOS Sharpe, and much lower drawdown.
- Weakness: turnover is higher than the Ridge strategies, so transaction costs matter.
- This should be presented as the lowest-coefficient-overfit candidate, not as proof of zero overfit.


## Lower-Overfit Final Algorithm

A fitted Ridge model can still overfit because its coefficients are learned from history. The final conservative algorithm is therefore the annual expanding walk-forward two-block Ridge, not the static best-looking variant. This is not a claim of zero overfit; it is a lower-overfit design with fixed features, fixed Ridge penalties, fixed retraining schedule, fixed position sizing, fixed expanding median edge filter, no external overlays, and no full-sample weight/grid search.

Algorithm summary:
- Core block: `mom_60`, `rev_5`, `curve_spread`, `curve_ratio`, `cot_mm_level`, `cot_pm_oi_level`.
- Physical block: `public_inventory_change`, `receipts_change`, `cgl_inventory_change`, `crush_surprise`, `crush_utilization`.
- Annual expanding walk-forward retraining; coefficients use only data before each retrain date.
- Fixed penalties: core alpha `25`, physical alpha `1000`.
- Fixed target: 20-day forward dollar PnL.
- Fixed edge filter: expanding one-day-lagged median prediction dispersion.


In [ ]:
from grain_futures_strategy import run_lower_overfit_strategy

lower_overfit_results = run_lower_overfit_strategy(DATA_DIR, split_date=SPLIT_DATE)
print("Lower-overfit strategy assumptions")
display(lower_overfit_results["assumptions"])
print("Lower-overfit strategy metrics")
display(lower_overfit_results["metrics"].round(3))


Lower-overfit read-through:
- This version is more conservative than the static edge-filtered Ridge because every trading-year prediction is generated by coefficients fitted only on prior data.
- It gives up some static full-period Sharpe, but it improves the honesty of the research claim.
- The static edge-filtered strategy and external overlays can remain research candidates or appendix extensions, but they should not be described as the lowest-overfit final strategy.
- The remaining required test is the untouched 2021-February 2026 holdout.


## Higher-Performing Cost-Aware Candidate

This section keeps the previously researched cost-aware blend as a higher-performing candidate, not as the lowest-overfit final algorithm. It uses a fixed 50/50 blend of two non-ML sleeves:
- 50% 2-day skip-rebalance: lower turnover and stronger OOS PnL after realistic costs.
- 50% multi-condition opportunity filter: higher Sharpe, better drawdown, and better crisis/regime behavior.

Because the opportunity-filter thresholds were selected during research, this candidate has more selection risk than the walk-forward final algorithm above. Present it as a performance candidate that must be frozen before the 2021-February 2026 holdout, not as proof of no overfit.


In [12]:
final_strategy_results = run_final_strategy_selection(DATA_DIR, split_date=SPLIT_DATE)
final_assumptions = final_strategy_results['assumptions']

print('Final strategy assumptions')
print('--------------------------')
for _, row in final_assumptions.iterrows():
    print(str(row['assumption']) + ': ' + str(row['value']))

final_assumptions


Final strategy assumptions
--------------------------
final_strategy: Fixed 50/50 blend: 2-day skip-rebalance + multi-condition opportunity filter
skip_rebalance_weight: 0.5
multi_condition_weight: 0.5
prediction_dispersion_quantile: 0.4
curve_dispersion_quantile: 0.4
momentum_dispersion_quantile: 0.4
threshold_policy: Expanding and one-day lagged; not retuned after holdout review
trade_cost_per_lot: 8.75
holding_cost_rate_annual: 0.05
margin_budget: inf
selection_policy: Blend weights are fixed, not optimized on Sharpe/PnL


                        assumption                                                                         value
0                   final_strategy  Fixed 50/50 blend: 2-day skip-rebalance + multi-condition opportunity filter
1            skip_rebalance_weight                                                                           0.5
2           multi_condition_weight                                                                           0.5
3   prediction_dispersion_quantile                                                                           0.4
4        curve_dispersion_quantile                                                                           0.4
5     momentum_dispersion_quantile                                                                           0.4
6                 threshold_policy                Expanding and one-day lagged; not retuned after holdout review
7               trade_cost_per_lot                                                              

In [13]:
final_metric_cols = [
    'window', 'trading_days', 'winning_days', 'losing_days', 'flat_days',
    'hit_rate', 'sharpe', 'total_pnl', 'gross_pnl', 'trade_cost',
    'holding_cost', 'total_cost', 'CDF', 'max_drawdown',
    'avg_daily_turnover', 'avg_gross_exposure', 'avg_margin_used',
    'avg_margin_scale'
]
final_metrics = final_strategy_results['final_strategy_metrics'][final_metric_cols]
final_metrics.round(3)


       window  trading_days  winning_days  losing_days  flat_days  hit_rate  sharpe  total_pnl  gross_pnl  trade_cost  holding_cost  total_cost   CDF  max_drawdown  avg_daily_turnover  avg_gross_exposure  avg_margin_used  avg_margin_scale
    in_sample          1302           657          645          0     0.505   0.811  11061.073  13225.725    1755.052       409.600    2164.651 0.164     -2845.121               0.154               0.620         1585.547               1.0
out_of_sample           637           331          306          0     0.520   1.529   8410.630   9596.388     953.822       231.935    1185.757 0.124     -2067.116               0.171               0.700         1835.092               1.0
  full_period          1939           988          951          0     0.510   1.014  19471.704  22822.112    2708.874       641.535    3350.409 0.147     -2845.121               0.160               0.646         1667.528               1.0


Cost-aware candidate read-through:
- The blend keeps turnover close to the 2-day skip-rebalance sleeve while adding crisis/high-conviction exposure from the multi-condition sleeve.
- The reported PnL is net of trade cost and holding/funding cost under the market-assumption case.
- This is no longer labelled as the lowest-overfit final strategy because the opportunity thresholds were part of the research process.
- If the extended holdout rejects the q=0.40 opportunity filter, the fallback should be the lower-overfit annual walk-forward Ridge or the 2-day skip-rebalance sleeve, not retuned thresholds.


## Strategy Selection

Recommended selection by use case:

| Use case | Selected strategy | Why |
|---|---|---|
| Main tradable candidate | Static edge-filtered two-block Ridge | Best balance of OOS PnL, interpretability, and already uses both Cargill datasets. |
| Anti-overfit validation | Annual walk-forward Ridge | Uses only past data at each annual retrain; lower performance but cleaner evidence. |
| Higher Sharpe / crisis avoidance | Multi-condition opportunity filter | Validation-supported; fixes US-China negative Sharpe; trades fewer days. |
| Higher OOS PnL and lower turnover | 2-day skip-rebalance static strategy | Raises OOS PnL and lowers turnover, but not confirmed by walk-forward. |
| Diversification overlay | 80% outright + 20% inter-commodity spread | Improves Sharpe/drawdown but reduces OOS PnL. |
| Rejected | ML, broad Ridge, calendar spreads, smoothing, carry/storage-only | Either overfit, weak OOS, or not robust across regimes. |

Natural regime weighting update:
- I tested continuous opportunity-weighted exposure and a trailing-performance ensemble to reduce hard-threshold overfit risk.
- They are conceptually cleaner, but weaker in this sample. Use them as robustness checks unless the extended holdout proves otherwise.


## Equity Curves For Selected Strategies

In [ ]:
bt_static = results['model_bt']
bt_wf = results['walk_forward_bt']
static_2d_pos = skip_rebalance_positions(results['model_positions'], 2)
bt_static_2d, _ = backtest_positions(static_2d_pos, results['futures_pnl'], COST_PER_LOT)
multi_pos = selected_positions['Multi-condition filter']
bt_multi, _ = backtest_positions(multi_pos, results['futures_pnl'], COST_PER_LOT)
plt.figure(figsize=(12, 5))
plt.plot(bt_static.index, bt_static['cum_pnl'], label='Static edge-filtered')
plt.plot(bt_wf.index, bt_wf['cum_pnl'], label='Annual walk-forward Ridge', alpha=0.85)
plt.plot(bt_static_2d.index, bt_static_2d['cum_pnl'], label='Static 2-day skip-rebalance', alpha=0.85)
plt.plot(bt_multi.index, bt_multi['cum_pnl'], label='Multi-condition filter', alpha=0.85)
plt.axvline(pd.Timestamp(SPLIT_DATE), color='black', linestyle='--', linewidth=1, label='train/test split')
plt.title('Selected strategy cumulative PnL')
plt.ylabel('Dollars per aggregate gross lot')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Optional External Macro Overlay: FX, Crude, And Rates

This cell is intentionally optional and separate from the selected strategy. It downloads external Yahoo Finance data for USD index, crude oil, Treasury yields, EUR/USD, and BRL/USD, then tests those lagged macro features as a strongly regularized overlay on the Cargill-aware model. Leave `USE_YFINANCE_MACRO_OVERLAY = False` unless you want to fetch external data and rerun the experiment.


In [ ]:
# Optional external macro overlay from yfinance.
# Flip this to True only when you want the notebook to download Yahoo Finance data.
USE_YFINANCE_MACRO_OVERLAY = False

if USE_YFINANCE_MACRO_OVERLAY:
    from macro_yfinance_experiment import run_macro_yfinance_experiment

    macro_yfinance_results = run_macro_yfinance_experiment()
    display(macro_yfinance_results['results'])
else:
    print('Set USE_YFINANCE_MACRO_OVERLAY = True to run the optional yfinance FX/crude/rates overlay backtest.')


## Optional External EIA Ethanol Overlay

This cell is optional and separate from the selected Cargill-aware strategy. It uses EIA weekly fuel ethanol production and ending stocks as a corn-demand overlay. It reads `EIA_API_KEY` / `EIA_KEY` from the environment, or the existing natgas config path if available. Leave `USE_EIA_ETHANOL_OVERLAY = False` unless you want to fetch EIA data and rerun the experiment.


In [ ]:
# Optional EIA ethanol production/stocks overlay.
# Uses EIA_API_KEY/EIA_KEY, or the natgas config path if present.
USE_EIA_ETHANOL_OVERLAY = False

if USE_EIA_ETHANOL_OVERLAY:
    from eia_ethanol_experiment import run_eia_ethanol_experiment

    eia_ethanol_results = run_eia_ethanol_experiment()
    display(eia_ethanol_results['results'])
else:
    print('Set USE_EIA_ETHANOL_OVERLAY = True to run the optional EIA ethanol overlay backtest.')


## Optional External Meteostat Weather Overlay

This cell is optional and separate from the selected Cargill-aware strategy. Meteostat is open weather data and does not require an API key. It tests shared and commodity-weighted crop-belt weather features, including temperature, CDD/HDD, precipitation, dryness, heat/freeze stress, GDD-style accumulation, and seasonal planting/growing/harvest interactions. Leave `USE_METEOSTAT_OVERLAY = False` unless you want to fetch the external weather data and rerun the expanded weather backtest.


In [ ]:
# Optional Meteostat weather overlay.
# Requires the meteostat package. It is open/no-key data, but it downloads external weather files.
USE_METEOSTAT_OVERLAY = False

if USE_METEOSTAT_OVERLAY:
    from meteostat_experiment import run_meteostat_experiment

    meteostat_results = run_meteostat_experiment()
    display(meteostat_results['results'])
else:
    print('Set USE_METEOSTAT_OVERLAY = True to run the optional Meteostat weather overlay backtest.')


## Optional Combined Macro + EIA Ethanol Overlay

This optional cell tests activating both external overlays together: yfinance macro markets plus EIA ethanol production/stocks. It also includes a no-grid version using fixed or lagged-only ensemble rules, so the Macro + EIA extension can be evaluated without selecting weights from the full result table. It is separate from the selected Cargill-aware strategy and is off by default because it fetches external data.


In [ ]:
# Optional combined external overlay: yfinance macro + EIA ethanol.
USE_COMBINED_EXTERNAL_OVERLAY = False

if USE_COMBINED_EXTERNAL_OVERLAY:
    from combined_external_nogrid_experiment import run_combined_external_nogrid_experiment
    from combined_external_experiment import run_combined_external_experiment

    combined_external_nogrid_results = run_combined_external_nogrid_experiment()
    print('No-grid Macro + EIA variants')
    display(combined_external_nogrid_results['results'])

    combined_external_results = run_combined_external_experiment()
    print('Reference Macro + EIA weight grid')
    display(combined_external_results['results'])
else:
    print('Set USE_COMBINED_EXTERNAL_OVERLAY = True to run no-grid Macro + EIA variants and the reference overlay grid.')


## Optional Combined Macro + EIA + Meteostat Weather Overlay

This optional cell tests the full external-data mix: yfinance macro markets, EIA ethanol production/stocks, and Meteostat commodity-weighted weather with CDD/HDD, precipitation, dryness, heat/freeze stress, and GDD-style features. It is separate from the selected Cargill-aware strategy and is off by default because it fetches external data.


In [ ]:
# Optional combined external overlay: yfinance macro + EIA ethanol + Meteostat weather.
USE_COMBINED_MACRO_EIA_WEATHER_OVERLAY = False

if USE_COMBINED_MACRO_EIA_WEATHER_OVERLAY:
    from combined_macro_eia_weather_experiment import run_combined_macro_eia_weather_experiment

    combined_macro_eia_weather_results = run_combined_macro_eia_weather_experiment()
    display(combined_macro_eia_weather_results['results'])
else:
    print('Set USE_COMBINED_MACRO_EIA_WEATHER_OVERLAY = True to run macro + EIA + Meteostat weather overlay grid.')


## Optional Combined Risk + Spread Diagnostics

This optional cell tests why the full Macro + EIA overlay has a worse drawdown, then compares conservative Macro/EIA weights, drawdown throttles, calendar spreads, and intercommodity spread overlays as separate rows. It is off by default because it fetches external data.


In [ ]:
# Optional drawdown and spread diagnostics for Macro + EIA.
USE_COMBINED_RISK_SPREAD_DIAGNOSTICS = False

if USE_COMBINED_RISK_SPREAD_DIAGNOSTICS:
    from combined_risk_spread_experiment import run_combined_risk_spread_experiment

    combined_risk_spread_results = run_combined_risk_spread_experiment()
    display(combined_risk_spread_results['results'])
else:
    print('Set USE_COMBINED_RISK_SPREAD_DIAGNOSTICS = True to run Macro + EIA drawdown/spread diagnostics.')


## Testability Of Next Ideas

| Idea | Testable now? | What is needed | How to test without overfitting |
|---|---|---|---|
| Add soybean meal/oil futures for a true tradable crush-spread sleeve | Not with the current train_set alone | Daily adjusted futures prices for soybean meal and soybean oil, aligned to this calendar; contract multipliers; realistic leg ratios and costs | Add the data as new files, define crush PnL from soybean/meal/oil legs, freeze the signal construction, and evaluate on the untouched 2021-February 2026 holdout |
| Hard multi-condition filter thresholds | Testable now, but overfit-prone | No new data needed to write the rule; extended data needed to score it | Treat q=0.40/0.40/0.40 as a validation-selected candidate, not guaranteed truth. Freeze it before holdout and report honestly even if it fails |
| Natural regime weighting without hard thresholds | Testable now | No new data needed | Use expanding lagged percentile opportunity scores or trailing performance weights. These avoid named-regime labels and hard cutoffs, but must still be judged on the holdout |

Important overfit note:
- Even q=0.40/0.40/0.40 can be overfit because it was selected after looking at the current sample and validation period.
- The lower-overfit alternative added here is continuous/natural weighting: exposure is scaled by expanding prediction, curve, and momentum opportunity percentiles, or blended by lagged trailing sleeve performance.
- In the current sample, these natural methods reduce threshold-selection risk but underperform the simpler 2-day skip-rebalance and the hard multi-condition filter.
- Therefore the notebook should present the hard multi-condition filter as a strong validation-selected candidate, and the natural weighting methods as robustness checks, not as proven final strategies.

Pre-registration plan for the next holdout:
- Freeze the candidate list before loading 2021-February 2026.
- Include: static edge-filtered, 2-day skip-rebalance, hard multi-condition q=0.40/0.40/0.40, natural opportunity-weighted exposure, trailing-performance natural ensemble.
- Primary metrics: OOS Sharpe, OOS PnL, full Sharpe, max drawdown, turnover, transaction-cost-adjusted Sharpe, and 2022 Ukraine-war period Sharpe.
- Do not change thresholds, lookbacks, or ensemble members after seeing the holdout results.

## Final Notes And Next Work

What would make the research stronger:
- Freeze and run the no-fit Cargill-aware reversion rule on the 2021-February 2026 extension.
- Freeze and run the annual walk-forward Ridge benchmark on the same extension without changing features, alphas, edge quantile, or retraining schedule.
- Add 2022 Ukraine war as a named diagnostic regime only after the extended data is available; do not use named regimes as trading inputs.
- Keep external Macro/EIA/weather overlays in the appendix unless they survive the untouched holdout.
- Add realistic transaction costs and margin/risk constraints to every final candidate, especially the no-fit reversion rule because turnover is higher.
- Add soybean meal/oil futures only if a true tradable crush-spread sleeve is required; it is not testable with the current train_set alone.


## Per-Grain No-Fit Strategy Check

This optional research cell tests whether fixed commodity-specific recipes improve the lower-overfit final rule. It keeps the recipes deterministic and uses no Ridge coefficients, but the research choice itself can still overfit, so the result is logged as an experiment rather than silently promoted.

The key comparison is per-grain independent outright positions versus relative-value market-neutral positions. In the current 2010-2020 train sample, independent outright recipes do not improve OOS performance; the robust edge remains cross-sectional grain reversion with a small Cargill/public physical tilt.


In [ ]:
# Optional per-grain no-fit experiment
# Leave this cell off for the final conservative strategy; run it to audit the rejected per-grain variants.
RUN_PER_GRAIN_NO_FIT_EXPERIMENT = False

if RUN_PER_GRAIN_NO_FIT_EXPERIMENT:
    from per_grain_no_fit_experiment import run_per_grain_no_fit_experiment

    per_grain_out = run_per_grain_no_fit_experiment()
    display(per_grain_out["results"].round(3))
    display(per_grain_out["asset_results"].round(3))

    best_zero_cost = per_grain_out["results"].query("cost_adjusted == False").iloc[0]
    best_cost_adjusted = per_grain_out["results"].query("cost_adjusted == True").iloc[0]
    print("Best zero-cost variant:", best_zero_cost["experiment"], "OOS Sharpe", round(best_zero_cost["oos_sharpe"], 3))
    print("Best cost-adjusted variant:", best_cost_adjusted["experiment"], "OOS Sharpe", round(best_cost_adjusted["oos_sharpe"], 3))
else:
    print("Per-grain no-fit experiment skipped. Set RUN_PER_GRAIN_NO_FIT_EXPERIMENT = True to run it.")


## Soybean-Only Lower-Overfit Strategy Check

This optional cell tests soybean as its own strategy rather than forcing one rule across corn, soybeans, and wheat. The strategy uses fixed recipes, no Ridge coefficients, and selects the candidate using only the pre-2018 train/validation split. The 2018-2020 period is reported as the test result.

The selected soybean-only candidate is a conservative long-only blend of trend/curve tightness, public/Cargill inventory pressure, and Cargill crush pressure. It is documented as an optional sleeve, not a replacement for the stronger all-grain cross-sectional book.


In [ ]:
# Optional soybean-only no-fit experiment
# This cell is off by default so it does not change the main strategy results.
RUN_SOYBEAN_ONLY_NO_FIT_EXPERIMENT = False

if RUN_SOYBEAN_ONLY_NO_FIT_EXPERIMENT:
    from soybean_no_fit_experiment import run_soybean_no_fit_experiment

    soybean_out = run_soybean_no_fit_experiment()
    display(soybean_out["results"].round(3))
    print("Selected by pre-2018 validation rule:")
    display(soybean_out["selected_by_validation"].to_frame("value"))

    selected_key = soybean_out["selected_by_validation"]["recipe"] + "_" + soybean_out["selected_by_validation"]["mode"]
    bt = soybean_out["backtests"][selected_key + "_cost_adjusted"]
    bt["cum_pnl"].plot(title="Soybean-only selected strategy, cost-adjusted cumulative PnL", figsize=(10, 4));
else:
    print("Soybean-only no-fit experiment skipped. Set RUN_SOYBEAN_ONLY_NO_FIT_EXPERIMENT = True to run it.")


In [ ]:
# Soybean Given-Data-Only Signal Cell
# Uses only signals present in the provided training files / source data.
from soybean_external_signal_experiment import run_soybean_given_only_experiment

soybean_given_out = run_soybean_given_only_experiment()
soybean_given_results = soybean_given_out["results"]
display(soybean_given_results.round(3))

soybean_given_best_zero = soybean_given_results.query("cost_adjusted == False").iloc[0]
soybean_given_best_cost = soybean_given_results.query("cost_adjusted == True").iloc[0]
print("Best given-data zero-cost:", soybean_given_best_zero["experiment"], "OOS Sharpe", round(soybean_given_best_zero["test_sharpe"], 3), "OOS DD", round(soybean_given_best_zero["test_max_drawdown"], 3))
print("Best given-data cost-adjusted:", soybean_given_best_cost["experiment"], "OOS Sharpe", round(soybean_given_best_cost["test_sharpe"], 3), "OOS DD", round(soybean_given_best_cost["test_max_drawdown"], 3))


In [ ]:
# Soybean External Signal Family Cell
# Uses optional free external data: yfinance soybean complex / FX / macro proxies and Meteostat HDD/CDD/weather.
from soybean_external_signal_experiment import run_soybean_external_signal_experiment

soybean_external_out = run_soybean_external_signal_experiment(include_weather=True)
if soybean_external_out["errors"]:
    print("External data warnings:")
    for warning in soybean_external_out["errors"]:
        print("-", warning)

soybean_external_results = soybean_external_out["results"]
display(soybean_external_results.round(3))

print("Pre-2018 overfit-control family selection:")
display(soybean_external_out["overfit_control_selection"].round(3))
print("Equal-family selected families:", soybean_external_out["overfit_control_selected_families"])

print("Relaxed fixed-weight pre-2018 selection menu:")
display(soybean_external_out["weight_relaxed_selection"].round(3))
print("Score-selected candidate:", soybean_external_out["weight_relaxed_selected_candidate"])
print("Drawdown-priority selected candidate:", soybean_external_out["drawdown_priority_selected_candidate"])

print("Observable regime-shift pre-2018 selection menu:")
display(soybean_external_out["regime_selection"].round(3))
print("Regime-selected candidate:", soybean_external_out["regime_selected_candidate"])

print("Fund-usable regime/drawdown blend menu:")
display(soybean_external_out["fund_selection"].round(3))
print("Fund-selected candidate:", soybean_external_out["fund_selected_candidate"])

if not soybean_external_results.empty:
    soybean_external_best_zero = soybean_external_results.query("cost_adjusted == False").iloc[0]
    soybean_external_best_cost = soybean_external_results.query("cost_adjusted == True").iloc[0]
    print("Best external zero-cost:", soybean_external_best_zero["experiment"], "OOS Sharpe", round(soybean_external_best_zero["test_sharpe"], 3), "OOS DD", round(soybean_external_best_zero["test_max_drawdown"], 3))
    print("Best external cost-adjusted:", soybean_external_best_cost["experiment"], "OOS Sharpe", round(soybean_external_best_cost["test_sharpe"], 3), "OOS DD", round(soybean_external_best_cost["test_max_drawdown"], 3))

    selected_rows = soybean_external_results[soybean_external_results["experiment"].str.contains("overfit_controlled_pre2018|weight_relaxed_pre2018|drawdown_priority_pre2018|regime_pre2018|fund_pre2018", regex=True)]
    display(selected_rows.round(3))


## Corn Strategy With EIA Ethanol

This optional cell runs the corn-only lower-overfit research track. It uses the provided Cargill inventory and crush/activity fields, EIA ethanol production/stocks, yfinance FX/macro proxies, and optional Meteostat weather. The recipes are fixed-weight and no Ridge coefficients are fitted.

The logged result favors a conservative long-only sleeve: `requirement_given_80_ethanol_10_fx_10`, which keeps Cargill/provided signals as the main block and uses ethanol as a small corn-demand overlay.


In [ ]:
# Corn external signal strategy cell
# Uses required Cargill inventory/crush fields plus EIA ethanol data.
from corn_external_signal_experiment import run_corn_signal_experiment

RUN_CORN_EXTERNAL_SIGNAL_EXPERIMENT = False

if RUN_CORN_EXTERNAL_SIGNAL_EXPERIMENT:
    corn_out = run_corn_signal_experiment(include_weather=True, include_eia=True)
    if corn_out["errors"]:
        print("Warnings:")
        for warning in corn_out["errors"]:
            print("-", warning)

    corn_results = corn_out["results"]
    display(corn_out["selection"].round(3))
    display(corn_results.round(3))
    print("Validation-selected candidate:", corn_out["selected_candidate"], corn_out["selected_mode"])

    recommended = corn_results[
        (corn_results["family"] == "requirement_given_80_ethanol_10_fx_10")
        & (corn_results["mode"] == "long_only")
        & (corn_results["cost_adjusted"] == True)
    ]
    if not recommended.empty:
        row = recommended.iloc[0]
        print(
            "Recommended corn sleeve:",
            row["family"], row["mode"],
            "OOS Sharpe", round(row["test_sharpe"], 3),
            "OOS DD", round(row["test_max_drawdown"], 3),
            "Full Sharpe", round(row["full_sharpe"], 3),
        )
else:
    print("Corn experiment skipped. Set RUN_CORN_EXTERNAL_SIGNAL_EXPERIMENT = True to fetch EIA/yfinance/Meteostat data and rerun it.")
